In [1]:
from pathlib import Path
import json
import re
import numpy as np
import mne
import logging

# Set up logging - must be at the top
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


In [11]:

# 1. Input/Output directories
# ----------------------------------------------------------------------
BIDS_ROOT = Path("/Volumes/cmvm/scs/groups/HELIOS-BD/Part B/hbd_vep")        
DERIV_DIR  = BIDS_ROOT / "derivatives" / "icalabel-vep"
DECODE_ROOT = Path("/Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP")
DECODE_DIR = DECODE_ROOT / "Luminance_NPZ"
#DECODE_DIR = DECODE_ROOT / "L_M_NPZ"
#DECODE_DIR = DECODE_ROOT / "S_cone_NPZ"
DECODE_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"BIDS root: {BIDS_ROOT}")
logger.info(f"Derivatives directory: {DERIV_DIR}")
logger.info(f"Decoding root: {DECODE_ROOT}")
logger.info(f"Decoding directory: {DECODE_DIR}")


2026-03-02 14:20:46,408 - INFO - BIDS root: /Volumes/cmvm/scs/groups/HELIOS-BD/Part B/hbd_vep
2026-03-02 14:20:46,409 - INFO - Derivatives directory: /Volumes/cmvm/scs/groups/HELIOS-BD/Part B/hbd_vep/derivatives/icalabel-vep
2026-03-02 14:20:46,410 - INFO - Decoding root: /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP
2026-03-02 14:20:46,412 - INFO - Decoding directory: /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ


In [12]:
import re

# List of subjects with age information available
SUBJECTS_WITH_AGE =[
    '1005','1007','1014','1004','1020','1017','1016','1002','1023','1034',
    '1008','1041','3011','1001','3005','1042','1039','2023','3007','3030',
    '3016','3014','2020','1011','1024','3001','1038','2006','1021','2009',
    '1010','3006','1046','2028','2026','2037','3027','1044','1028','3034',
    '2017','1052','3039','3041','1026','1018','2029','2002','3008'
]

SUBJECTS = sorted(
    d.name.replace("sub-", "")
    for d in DERIV_DIR.glob("sub-*")
    if (
        d.is_dir()
        and re.fullmatch(r"sub-\d{4}", d.name)
        and d.name.replace("sub-", "") in SUBJECTS_WITH_AGE
    )
)
logger.info(f"Found {len(SUBJECTS)} subjects: {SUBJECTS}")

2026-03-02 14:20:56,436 - INFO - Found 49 subjects: ['1001', '1002', '1004', '1005', '1007', '1008', '1010', '1011', '1014', '1016', '1017', '1018', '1020', '1021', '1023', '1024', '1026', '1028', '1034', '1038', '1039', '1041', '1042', '1044', '1046', '1052', '2002', '2006', '2009', '2017', '2020', '2023', '2026', '2028', '2029', '2037', '3001', '3005', '3006', '3007', '3008', '3011', '3014', '3016', '3027', '3030', '3034', '3039', '3041']


In [13]:
# 3. Electrode-set menu 
# ----------------------------------------------------------------------
ELECTRODE_SETS = {
    "1": {
        "name": "64", 
        "channels": list(range(1, 65)),
        "description": "All 64 electrodes"
    },
    "2": {
        "name": "PO", 
        "channels": [29, 27, 64, 25, 26, 30, 63, 62, 28, 24,
                     23, 22, 21, 20, 31, 57, 58, 59, 60, 61],
        "description": "Posterior electrodes only"
    },
    "3": {
        "name": "32", 
        "channels": [1, 34, 3, 36, 7, 5, 38, 40, 42, 9, 11, 46,
                     44, 15, 13, 48, 50, 52, 17, 19, 56, 54, 23, 21, 31, 58, 60, 26, 63, 27, 29, 64],
        "description": "32-electrode grid"
    },
}

print("Electrode options:")
print("  1- All 64 electrodes")
print("  2- Posterior only")
print("  3- 32-electrode grid")

while True:
    sel = input("Choose electrode set [1/2/3]: ").strip()
    if sel in ELECTRODE_SETS:
        ELEC_CFG = ELECTRODE_SETS[sel]
        break
    print(f"Invalid selection: {sel}. Please choose 1, 2, or 3.")

logger.info(f"Selected electrode set: {ELEC_CFG['name']} - {ELEC_CFG['description']}")
logger.info(f"Channels: {len(ELEC_CFG['channels'])}")

Electrode options:
  1- All 64 electrodes
  2- Posterior only
  3- 32-electrode grid


2026-03-02 14:21:20,660 - INFO - Selected electrode set: 64 - All 64 electrodes
2026-03-02 14:21:20,661 - INFO - Channels: 64


In [14]:

# 4. Stimulus-dimension menu 
# ----------------------------------------------------------------------
ANALYSES = {
    "1": {  # luminance
        "name": "luminance",
        "event_types": [1, 2, 3, 4],
        "labels": ["lum1", "lum2", "lum3", "lum4"],
        "plot_colors": ["#d9d9d9", "#999999", "#666666", "#262626"],
        "description": "Luminance analysis"
    },
    "2": {  # L–M
        "name": "L-M",
        "event_types": [5, 6, 7, 8],
        "labels": ["LM1", "LM2", "LM3", "LM4"],
        "plot_colors": ["#d98c8c", "#b23333", "#800000", "#260000"],
        "description": "L-M (red-green) analysis"
    },
    "3": {  # S-cone
        "name": "S-cone",
        "event_types": [9, 10, 11, 12],
        "labels": ["S1", "S2", "S3", "S4"],
        "plot_colors": ["#9cbfff", "#6666cc", "#333399", "#000026"],
        "description": "S-cone (blue-yellow) analysis"
    },
}

print("\nDecode which stimulus dimension?")
print("  1- Luminance")
print("  2- L minus M")
print("  3- S-cone")

while True:
    analysis_choice = input("Enter your choice [1/2/3]: ").strip()
    if analysis_choice in ANALYSES:
        ANALYSIS_CFG = ANALYSES[analysis_choice]
        break
    print(f"Invalid selection: {analysis_choice}. Please choose 1, 2, or 3.")

logger.info(f"Selected analysis: {ANALYSIS_CFG['name']} - {ANALYSIS_CFG['description']}")
logger.info(f"Event types: {ANALYSIS_CFG['event_types']}")
logger.info(f"Labels: {ANALYSIS_CFG['labels']}")



Decode which stimulus dimension?
  1- Luminance
  2- L minus M
  3- S-cone


2026-03-02 14:21:36,451 - INFO - Selected analysis: luminance - Luminance analysis
2026-03-02 14:21:36,452 - INFO - Event types: [1, 2, 3, 4]
2026-03-02 14:21:36,452 - INFO - Labels: ['lum1', 'lum2', 'lum3', 'lum4']


In [16]:
CONFIG_PATH = DECODE_DIR / "decode_config.json"

config_data = {
    "bids_root": str(BIDS_ROOT),
    "deriv_dir": str(DERIV_DIR),
    "decode_dir": str(DECODE_DIR),
    "subjects": list(SUBJECTS) if isinstance(SUBJECTS, set) else SUBJECTS,
    "electrode_set": list(ELEC_CFG) if isinstance(ELEC_CFG, set) else ELEC_CFG,
    "analysis": ANALYSIS_CFG,
}

with open(CONFIG_PATH, 'w') as f:
    json.dump(config_data, f, indent=2)
    logger.info(f"Configuration saved to {CONFIG_PATH}")

2026-03-02 14:22:04,413 - INFO - Configuration saved to /Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/Luminance_NPZ/decode_config.json


In [17]:

# 6. Get epochs file path for a subject
# ----------------------------------------------------------------------
def epochs_path_for(subject):
    eeg_dir = DERIV_DIR / f"sub-{subject}" / "eeg"
    fname   = f"sub-{subject}_task-vep_proc-clean_epo.fif"
    path    = eeg_dir / fname
    
    if not path.exists():
        raise FileNotFoundError(f"Epochs file not found: {path}")
    
    return path


In [18]:

# 7. Process each subject and save NPZ files
# ----------------------------------------------------------------------
for subject in SUBJECTS:
    logger.info(f"\n{'='*60}")
    logger.info(f"Processing subject: {subject}")
    logger.info(f"Electrode set: {ELEC_CFG['name']}")
    logger.info(f"Analysis: {ANALYSIS_CFG['name']}")
    
    try:
        # Load epochs
        epo_path = epochs_path_for(subject)
        logger.info(f"Loading epochs from: {epo_path}")
        epochs = mne.read_epochs(epo_path, preload=True, verbose=False)
        
        # Log initial info
        logger.info(f"Original epochs info: {len(epochs)} trials, {len(epochs.ch_names)} channels, {len(epochs.times)} time points")
        logger.info(f"Event codes present: {np.unique(epochs.events[:, 2])}")
        
        # Select channels - convert 1-based indices to 0-based
        ch_indices = [idx - 1 for idx in ELEC_CFG["channels"]]
        
        # Verify indices are within range
        if max(ch_indices) >= len(epochs.ch_names):
            logger.error(f"Channel index {max(ch_indices)+1} out of range for subject {subject}")
            raise IndexError("Channel index out of range")
        
        epochs.pick(ch_indices)
        ch_names = epochs.ch_names
        
        logger.info(f"Selected {len(ch_names)} channels: {ch_names[:3]}...{ch_names[-3:]}")
        
        # Filter events
        keep_codes = ANALYSIS_CFG["event_types"]
        mask = np.isin(epochs.events[:, 2], keep_codes)
        epochs = epochs[mask]
        
        if len(epochs) == 0:
            logger.error(f"No epochs found for event codes {keep_codes}")
            raise ValueError("No epochs after filtering")
        
        # Verify we have all event types
        found_codes = np.unique(epochs.events[:, 2])
        missing_codes = set(keep_codes) - set(found_codes)
        
        if missing_codes:
            logger.warning(f"Missing event codes: {missing_codes}")
        
        logger.info(f"After filtering: {len(epochs)} trials")
        logger.info(f"Found event codes: {found_codes}")
        
        # Create labels
        code_to_label = {code: i for i, code in enumerate(keep_codes)}
        y = np.array([code_to_label[code] for code in epochs.events[:, 2]], dtype=np.int32)
        
        # Log class distribution
        unique_labels, counts = np.unique(y, return_counts=True)
        for label, count in zip(unique_labels, counts):
            logger.info(f"Class {label}: {count} trials")
        
        # Prepare data for saving
        X = epochs.get_data()  # Shape: (n_trials, n_channels, n_times)
        times = epochs.times.astype(np.float32)
        
        # Verify data dimensions
        assert X.shape[0] == len(y), "Trial count mismatch between X and y"
        assert X.shape[1] == len(ch_names), "Channel count mismatch"
        assert X.shape[2] == len(times), "Time point mismatch"
        
        # Create output filename
        out_file = DECODE_DIR / f"sub-{subject}_{ANALYSIS_CFG['name']}_{ELEC_CFG['name']}_data.npz"
        
        # Save data with explicit array names and proper data types
        np.savez_compressed(
            out_file,
            X=X.astype(np.float32),       # EEG data (float32 for efficiency)
            y=y,                          # Class labels (int32)
            times=times,                  # Time points (float32)
            ch_names=np.array(ch_names),  # Channel names (1D array)
            event_codes=epochs.events[:, 2],  # Original event codes
            subject=subject,
            electrode_set=ELEC_CFG["name"],
            analysis=ANALYSIS_CFG["name"]
        )
        
        logger.info(f"Saved NPZ file: {out_file}")
        logger.info(f"Data shape: {X.shape} (trials × channels × time)")
        logger.info(f"Time range: {times[0]:.3f}s to {times[-1]:.3f}s")
        
    except Exception as e:
        logger.error(f"Error processing subject {subject}: {str(e)}", exc_info=True)
        continue

logger.info("\nProcessing complete!")
logger.info(f"NPZ files saved to: {DECODE_DIR}")

2026-03-02 14:22:17,779 - INFO - 
2026-03-02 14:22:17,780 - INFO - Processing subject: 1001
2026-03-02 14:22:17,781 - INFO - Electrode set: 64
2026-03-02 14:22:17,781 - INFO - Analysis: luminance
2026-03-02 14:22:17,858 - INFO - Loading epochs from: /Volumes/cmvm/scs/groups/HELIOS-BD/Part B/hbd_vep/derivatives/icalabel-vep/sub-1001/eeg/sub-1001_task-vep_proc-clean_epo.fif
2026-03-02 14:22:19,483 - INFO - Original epochs info: 716 trials, 69 channels, 256 time points
2026-03-02 14:22:19,483 - INFO - Event codes present: [ 1  2  3  4  5  6  7  8  9 10 11 12]
2026-03-02 14:22:19,495 - INFO - Selected 64 channels: ['Fp1', 'AF7', 'AF3']...['PO8', 'PO4', 'O2']
2026-03-02 14:22:19,514 - INFO - After filtering: 239 trials
2026-03-02 14:22:19,514 - INFO - Found event codes: [1 2 3 4]
2026-03-02 14:22:19,514 - INFO - Class 0: 60 trials
2026-03-02 14:22:19,514 - INFO - Class 1: 60 trials
2026-03-02 14:22:19,515 - INFO - Class 2: 60 trials
2026-03-02 14:22:19,515 - INFO - Class 3: 59 trials
2026-0

In [10]:
import numpy as np

# Load one of the generated files
data = np.load("/Users/farjam/OneDrive - University of Edinburgh/wellcome/Amir/New_Decoding_VEP/S_cone_NPZ/sub-1001_S-cone_64_data.npz", allow_pickle=True)

# Check keys and shapes
print("Keys:", list(data.keys()))
print("X shape:", data['X'].shape)
print("y shape:", data['y'].shape)
print("Times shape:", data['times'].shape)
print("Channel names:", data['ch_names'])
print("Event codes:", np.unique(data['event_codes']))
print("Subject:", data['subject'])
print("Electrode set:", data['electrode_set'])
print("Analysis:", data['analysis'])

Keys: ['X', 'y', 'times', 'ch_names', 'event_codes', 'subject', 'electrode_set', 'analysis']
X shape: (239, 64, 256)
y shape: (239,)
Times shape: (256,)
Channel names: ['Fp1' 'AF7' 'AF3' 'F1' 'F3' 'F5' 'F7' 'FT7' 'FC5' 'FC3' 'FC1' 'C1' 'C3'
 'C5' 'T7' 'TP7' 'CP5' 'CP3' 'CP1' 'P1' 'P3' 'P5' 'P7' 'P9' 'PO7' 'PO3'
 'O1' 'Iz' 'Oz' 'POz' 'Pz' 'CPz' 'Fpz' 'Fp2' 'AF8' 'AF4' 'AFz' 'Fz' 'F2'
 'F4' 'F6' 'F8' 'FT8' 'FC6' 'FC4' 'FC2' 'FCz' 'Cz' 'C2' 'C4' 'C6' 'T8'
 'TP8' 'CP6' 'CP4' 'CP2' 'P2' 'P4' 'P6' 'P8' 'P10' 'PO8' 'PO4' 'O2']
Event codes: [ 9 10 11 12]
Subject: 1001
Electrode set: 64
Analysis: S-cone
